# CNN: quando a posição do padrão não importa

**Para quem está começando.** Este notebook supõe que você já viu o MLP, da mesma pasta. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. criar um problema em que o mesmo padrão aparece em lugares diferentes;
2. treinar uma rede densa e uma convolucional e comparar as duas;
3. contar os parâmetros das duas;
4. rodar num dado de imagem de verdade.

Roda no Google Colab, que já traz o PyTorch. No Jupyter do seu computador, vale o ambiente do `README.md` (a versão usada aqui é `torch==2.13.0`).


## O que a convolução faz de diferente

Numa camada densa, cada neurônio da saída tem um peso para cada pixel da entrada. Isso traz dois problemas em imagem: o número de pesos explode com o tamanho da imagem, e o padrão que o neurônio aprendeu fica amarrado àquela posição. Se a mesma mancha aparece dez pixels à esquerda, ele tem de aprender tudo de novo.

Uma camada convolucional faz diferente. Ela tem um **kernel** (aqui, uma matriz 3x3) e o desliza pela imagem inteira, sempre com os mesmos pesos. O resultado é um mapa que marca onde o padrão do kernel aparece. Três consequências:

- **os pesos são compartilhados**: o mesmo detector vale para qualquer posição, e o número de parâmetros não depende do tamanho da imagem;
- **cada canal aprende um detector diferente**: as primeiras camadas costumam aprender bordas e cantos, as seguintes combinam isso em partes maiores, e isso vale para quase todo problema de imagem;
- **o `MaxPool`** reduz a resolução guardando a resposta mais forte de cada região. Ele dá ao modelo uma folga de posição e corta o custo das camadas seguintes.

O que a convolução **não** é: uma pílula de invariância total. Um kernel 3x3 com pool 2 tolera pequenos deslocamentos, não rotações nem mudanças de escala. Na prática, quem resolve isso é aumento de dados, e é por isso que bibliotecas de imagem existem.


## Passo 0: ambiente e configuração

A semente é fixada em `numpy` e em `torch`, e as funções de treino e de predição são as mesmas do notebook do MLP, adaptadas para receber imagem em vez de tabela.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (9, 4)
print(f"✅ torch {torch.__version__} em {device} | seed {RANDOM_STATE}")

## Passo 1: um padrão que muda de lugar

O conjunto é sintético, feito aqui mesmo: cada imagem 28x28 tem uma barra, horizontal ou vertical, em posição aleatória. A classe é a orientação da barra.

O detalhe que faz o experimento: o **treino** só tem barras numa região da imagem (a metade superior esquerda), e existem **dois testes**. Um na mesma região, outro em posições que o treino nunca viu. Se o modelo aprendeu "a barra está no canto", o segundo teste desmonta; se aprendeu "existe uma barra horizontal ou vertical", o segundo teste passa igual.


In [ ]:
def make_bars(n: int, size: int = 28, region: tuple[int, int] | None = None):
    """One bar per image, horizontal (0) or vertical (1), at a random position."""
    imagens = np.zeros((n, 1, size, size), dtype=np.float32)
    rotulos = rng.integers(0, 2, size=n)
    for i, rotulo in enumerate(rotulos):
        limites = region if region is not None else (2, size - 12)
        topo = int(rng.integers(*limites))
        esquerda = int(rng.integers(*limites))
        if rotulo == 0:
            imagens[i, 0, topo, esquerda:esquerda + 8] = 1.0
        else:
            imagens[i, 0, topo:topo + 8, esquerda] = 1.0
    return imagens, rotulos.astype(np.float32)


REGIAO_TREINO = (2, 10)          # metade superior esquerda
REGIAO_DESLOCADA = (16, 24)      # metade inferior direita, que o treino nao ve

X_cnn, y_cnn = make_bars(800, region=REGIAO_TREINO)
X_mesma, y_mesma = make_bars(300, region=REGIAO_TREINO)
X_deslocada, y_deslocada = make_bars(300, region=REGIAO_DESLOCADA)
print(f"🔎 treino {X_cnn.shape} | teste na mesma regiao {X_mesma.shape} "
      f"| teste deslocado {X_deslocada.shape}")

fig, eixos = plt.subplots(1, 4, figsize=(11, 3))
for eixo, indice in zip(eixos, [0, 1]):
    eixo.imshow(X_cnn[indice, 0], cmap="gray")
    eixo.set_title(f"treino, classe {int(y_cnn[indice])}")
for eixo, indice in zip(eixos[2:], [0, 1]):
    eixo.imshow(X_deslocada[indice, 0], cmap="gray")
    eixo.set_title(f"deslocado, classe {int(y_deslocada[indice])}")
for eixo in eixos:
    eixo.axis("off")
plt.tight_layout()
plt.show()

## Passo 2: as duas redes, com o mesmo treino

A rede densa recebe a imagem achatada, com 784 entradas. A convolucional recebe a imagem 28x28 com um canal e aplica dois blocos de convolução com `MaxPool`, antes de uma camada densa final. As duas terminam num único número, e a mesma função de perda do MLP transforma isso em probabilidade.

Repare na contagem de parâmetros: é ali que a diferença de desenho aparece, antes de qualquer resultado.


In [ ]:
class Densa(nn.Module):
    """Baseline: flatten the image and use dense layers."""

    def __init__(self, pixels: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(pixels, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x)


class Convolucional(nn.Module):
    """Two convolution blocks with pooling, then a dense head."""

    def __init__(self, size: int = 28):
        super().__init__()
        self.blocos = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
        )
        # Pool global: o mapa inteiro vira um valor por canal, entao o cabecalho nao ve posicao.
        self.cabeca = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(16, 1))

    def forward(self, x):
        return self.cabeca(self.blocos(x))


def contar_parametros(modelo: nn.Module) -> int:
    return sum(p.numel() for p in modelo.parameters())


print(f"parâmetros da densa: {contar_parametros(Densa(28 * 28)):,}")
print(f"parâmetros da convolucional: {contar_parametros(Convolucional(28)):,}")

In [ ]:
def to_tensor(X, y=None):
    """Numpy in, float32 tensor out; images keep the (batch, channel, height, width) shape."""
    X_t = torch.tensor(np.asarray(X, dtype=np.float32))
    if y is None:
        return X_t
    return X_t, torch.tensor(np.asarray(y, dtype=np.float32)).view(-1, 1)


def predict_proba(modelo: nn.Module, X) -> np.ndarray:
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(modelo(to_tensor(X))).numpy().ravel()


def train_model(modelo: nn.Module, X, y, epochs: int = 300, lr: float = 0.01):
    """Full-batch training; returns the loss of every epoch."""
    X_t, y_t = to_tensor(X, y)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    perdas = []
    for _ in range(epochs):
        modelo.train()
        otimizador.zero_grad()
        loss = loss_fn(modelo(X_t), y_t)
        loss.backward()
        otimizador.step()
        perdas.append(float(loss.item()))
    return perdas


def acuracia(modelo: nn.Module, X, y) -> float:
    return float(((predict_proba(modelo, X) > 0.5) == y).mean())


torch.manual_seed(RANDOM_STATE)
densa = Densa(28 * 28)
torch.manual_seed(RANDOM_STATE)
convolucional = Convolucional(28)

inicio = time.perf_counter()
train_model(densa, X_cnn, y_cnn)
train_model(convolucional, X_cnn, y_cnn)
print(f"⏱️ as duas redes juntas em {time.perf_counter() - inicio:.1f}s")

tabela = pd.DataFrame([
    {"modelo": "Densa", "parametros": contar_parametros(densa),
     "acuracia_mesma_regiao": acuracia(densa, X_mesma, y_mesma),
     "acuracia_deslocado": acuracia(densa, X_deslocada, y_deslocada)},
    {"modelo": "Convolucional", "parametros": contar_parametros(convolucional),
     "acuracia_mesma_regiao": acuracia(convolucional, X_mesma, y_mesma),
     "acuracia_deslocado": acuracia(convolucional, X_deslocada, y_deslocada)},
]).set_index("modelo")
display(tabela.round(4))

### Como ler isso

A coluna que decide é `acuracia_deslocado`. A rede densa aprendeu o padrão amarrado à posição: cada neurônio da primeira camada tem um peso por pixel, e o que ele aprendeu vale para aquele canto da imagem. Quando a barra se muda, os pesos certos não são acionados, e a acurácia desaba. Repare que ela fica **abaixo** do chute: a rede não fica neutra diante do que não conhece, ela responde errado com confiança.

Na convolucional há duas coisas somadas, e vale separá-las:

- a **convolução** compartilha o kernel entre posições, o que dá tolerância a deslocamento pequeno, da ordem do kernel mais o pool. Se o deslocamento for grande, ela sozinha ainda cai;
- o **pool global** (`AdaptiveMaxPool2d(1)`) resume o mapa inteiro num valor por canal, e aí a posição deixa de existir para o cabeçalho. É isso que faz a barra deslocada acionar o mesmo detector.

Some a coluna `parametros`: a convolucional faz mais com muito menos pesos, e o número não muda se a imagem crescer de 28x28 para 280x280, o que não é verdade para a densa.

Duas linhas para experimentar: troque o `AdaptiveMaxPool2d(1)` por `Flatten` (a versão sem invariância global, que reconstrói a posição no cabeçalho) e veja a acurácia no teste deslocado cair; e remova o `MaxPool2d` do meio, para ver o custo subir sem ganho.


## Passo 3: num dado de imagem de verdade

Agora o conjunto de dígitos do scikit-learn: 1 797 imagens 8x8 de dígitos escritos à mão, dez classes. Como o MLP da pasta responde sim ou não, aqui o cabeçalho muda: são dez saídas, e a perda passa a ser a de entropia cruzada para várias classes.

As duas redes usam a mesma divisão. A convolucional recebe a imagem 8x8 e a densa recebe os 64 pixels achatados.


In [ ]:
digitos = load_digits()
X_dig = (digitos.images / 16.0).astype(np.float32)[:, None, :, :]
y_dig = digitos.target.astype(np.int64)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_dig, y_dig, test_size=0.25, stratify=y_dig, random_state=RANDOM_STATE
)
print(f"🔎 digitos: treino {X_treino.shape} | teste {X_teste.shape} | "
      f"{len(np.unique(y_dig))} classes")


class ConvolucionalDigitos(nn.Module):
    """One convolution block, ten outputs: at 8x8 a second pool would empty the image."""

    def __init__(self):
        super().__init__()
        self.blocos = nn.Sequential(nn.Conv2d(1, 8, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2))
        self.cabeca = nn.Sequential(nn.Flatten(), nn.Linear(8 * 3 * 3, 10))

    def forward(self, x):
        return self.cabeca(self.blocos(x))


class DensaDigitos(nn.Module):
    """Dense baseline on the flattened pixels, ten outputs."""

    def __init__(self, pixels: int = 64, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(pixels, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 10))

    def forward(self, x):
        return self.net(x)


def treinar_digitos(modelo, epochs: int = 200, lr: float = 0.01):
    X_t = torch.tensor(X_treino)
    y_t = torch.tensor(y_treino)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    perdas = []
    for _ in range(epochs):
        modelo.train()
        otimizador.zero_grad()
        loss = loss_fn(modelo(X_t), y_t)
        loss.backward()
        otimizador.step()
        perdas.append(float(loss.item()))
    modelo.eval()
    with torch.no_grad():
        acertos = (modelo(torch.tensor(X_teste)).argmax(dim=1) == torch.tensor(y_teste)).float().mean()
    return perdas, float(acertos)

In [ ]:
torch.manual_seed(RANDOM_STATE)
modelos_digitos = {"Densa (64-32-10)": DensaDigitos(), "Convolucional": ConvolucionalDigitos()}

linhas = []
for nome, modelo in modelos_digitos.items():
    inicio = time.perf_counter()
    perdas, acc = treinar_digitos(modelo)
    linhas.append({"modelo": nome, "parametros": contar_parametros(modelo),
                   "acuracia_teste": acc, "treino_s": time.perf_counter() - inicio})

pd.DataFrame(linhas).set_index("modelo").round(4).pipe(display)

### Como ler isso

Nos dígitos 8x8 as duas redes ficam perto, e isso tem explicação: com 64 pixels, o kernel 3x3 cobre quase um quarto da imagem, e um segundo bloco de pool deixaria a imagem vazia (repare que a convolucional dos dígitos tem um bloco só, contra dois no Passo 2). É o oposto do cenário em que ela ganha, que é imagem grande com padrão pequeno e repetido.

O que continua valendo mesmo aqui é a economia de parâmetros e a tolerância a deslocamento, que apareceram no Passo 2. Guarde a regra: convolução é para dado em que a posição do padrão não carrega informação, e o custo dela aparece quando a imagem cresce.

## Resumo

A convolução troca "um peso por pixel" por "um kernel deslizante com pesos compartilhados". O efeito é o que o Passo 2 mediu: o mesmo detector vale em qualquer posição, o número de parâmetros não cresce com o tamanho da imagem, e o modelo para de depender de onde o padrão está. Pooling dá folga de posição e reduz o custo das camadas seguintes.

E a pílula de invariância não existe: convolução tolera pequenos deslocamentos, não rotação nem escala. Quem cobre isso é aumento de dados.

**Próximo passo:** o notebook da RNN, na mesma pasta, onde a entrada tem comprimento variável e a ordem dos elementos carrega a informação.

### Referências

**A convolução aplicada a imagem**

LeCun, Y., Boser, B., Denker, J. S., Henderson, D., Howard, R. E., Hubbard, W., & Jackel, L. D. (1989). *Backpropagation applied to handwritten zip code recognition.* Neural Computation 1(4), 541-551.

Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). *ImageNet classification with deep convolutional neural networks.* Advances in Neural Information Processing Systems (NeurIPS).

**O contexto**

Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.* MIT Press. Capítulo 9, sobre redes convolucionais e as razões do compartilhamento de pesos.

```bibtex
@article{lecun1989backpropagation,
  title   = {Backpropagation applied to handwritten zip code recognition},
  author  = {LeCun, Yann and Boser, Bernhard and Denker, John S. and Henderson, Donnie and
             Howard, Richard E. and Hubbard, Wayne and Jackel, Lawrence D.},
  journal = {Neural Computation},
  volume  = {1},
  number  = {4},
  pages   = {541--551},
  year    = {1989}
}

@inproceedings{krizhevsky2012imagenet,
  title     = {ImageNet classification with deep convolutional neural networks},
  author    = {Krizhevsky, Alex and Sutskever, Ilya and Hinton, Geoffrey E.},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  year      = {2012}
}

@book{goodfellow2016deep,
  title     = {Deep Learning},
  author    = {Goodfellow, Ian and Bengio, Yoshua and Courville, Aaron},
  year      = {2016},
  publisher = {MIT Press}
}
```
